# Relevance Pursuit: Research Directions

- Contributors: SebastianAment
- Last updated: September 8, 2026
- BoTorch version: 0.18.1+ (main, commit `d4b9fc655`)

> **Work in progress.** This notebook collects research around Relevance Pursuit that
> did not make it into the released BoTorch APIs. The cells are committed **without
> outputs and have not been executed against the current BoTorch release** — expect to
> fix things. It is shared so that others can pick up whichever thread is useful to
> them. Every section ends with a "Status & open problems" note describing what does
> not work yet.

## Background

Relevance Pursuit is a family of greedy algorithms for fitting models with an
*exactly sparse* parameter. The recipe is:

1. Parameterize the model so that setting a parameter to exactly zero removes the
   corresponding component from the model, and so that the gradient at zero is
   well-defined.
2. Maintain a *support*: the subset of indices that are currently allowed to be
   non-zero.
3. Alternate between (a) optimizing the marginal likelihood over the current support
   and (b) growing or shrinking the support by one or more elements.
4. Select among the resulting sequence of models — the *model trace* — by Bayesian
   model comparison.

Steps 2–4 are generic, and BoTorch implements them in
`botorch.models.relevance_pursuit` via `RelevancePursuitMixin`,
`forward_relevance_pursuit`, `backward_relevance_pursuit`, and
`get_posterior_over_support`. Step 1 is model-specific.

BoTorch currently ships a single instantiation of step 1: `SparseOutlierNoise`, which
puts the sparse parameter on per-observation noise variances and yields a robust GP
(see `botorch.models.robust_relevance_pursuit_model` and the
`relevance_pursuit_robust_regression` tutorial). For details, see
[Ament et al., *Robust Gaussian Processes via Relevance Pursuit*, NeurIPS 2024](https://arxiv.org/abs/2410.24222).

This notebook explores other places to point the same machinery, plus some theory
about when it works.

| Section | Sparse parameter | What it buys you |
|---|---|---|
| [A](#a) | ARD inverse squared lengthscales | Exact feature selection in high dimensions |
| [B](#b) | Inducing point precisions | Automatic selection of the number and location of inducing points |
| [C](#c) | — (analysis) | Why the convex parameterization of `SparseOutlierNoise` exists |
| [D](#d) | — (analysis) | A fixed-point iteration for the noise variance |
| [E](#e) | — (analysis) | Why the design distribution controls support recovery in A |

Sections are independent; skip to whichever is relevant.

In [ ]:
import copy
import math
from abc import ABC, abstractmethod

import matplotlib.pyplot as plt
import torch

from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from botorch.utils.constraints import NonTransformedInterval
from gpytorch.kernels import Kernel, RBFKernel, ScaleKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.means import ZeroMean
from gpytorch.mlls import ExactMarginalLogLikelihood
from torch import Tensor
from torch.nn import Parameter

from botorch.models.relevance_pursuit import RelevancePursuitMixin

torch.set_default_dtype(torch.double)

<a id="a"></a>
# A. Sparse feature selection: the `SparseFeatureKernel`

## Motivation

Automatic relevance determination (ARD) is the standard way to do feature selection
with a GP: give each input dimension its own lengthscale and let the marginal
likelihood shrink the irrelevant ones. A stationary ARD kernel is

$$k(x, x') = f\!\left(\sum_{i=1}^d \frac{(x_i - x_i')^2}{\ell_i^2}\right).$$

"Dimension $i$ is irrelevant" corresponds to $\ell_i \to \infty$, which an optimizer
can only approach asymptotically. There is no finite parameter value that expresses
*exactly* irrelevant, so ARD produces a soft ranking rather than a support, and any
threshold on $\ell_i$ is arbitrary.

Reparameterizing by $\beta_i = \ell_i^{-2}$ fixes this:

$$k(x, x') = f\!\left(\sum_{i=1}^d \beta_i (x_i - x_i')^2\right).$$

Now $\beta_i = 0$ removes dimension $i$ exactly, it lies at the boundary of the
feasible region rather than at infinity, and — importantly — the gradient of the
marginal likelihood with respect to $\beta_i$ at $\beta_i = 0$ is finite and
informative. That is exactly the interface `RelevancePursuitMixin` expects, so the
support machinery applies directly.

A remark on why this particular form. One might instead try to sparsify a dot-product
kernel, $k(x, x') = f(\sum_i x_i x_i' \beta_i)$. That does not work as well here: the
sparse solution $\beta_i = 0$ is a *stationary point* of that objective, so the
support-expansion criterion — which is the gradient at zero — is uninformative. The
squared-distance form does not have this problem.

Note also that we need `NonTransformedInterval` rather than a softplus or log
transform for the constraint, for the same reason it is required in
`SparseOutlierNoise`: smooth transforms map the interior of the parameter space onto
the interior of the constraint set, so exact zeros are not representable.

## Distances and kernel functions

We need all-pairs squared distances kept *per dimension*, since the sparse parameter
weights them individually.

In [ ]:
def all_distances(x1: Tensor, x2: Tensor) -> Tensor:
    """All-pairs squared distances, per dimension, differentiable everywhere.

    Args:
        x1: `n x d` tensor of data.
        x2: `m x d` tensor of data.

    Returns:
        A `n x m x d` tensor of per-dimension squared differences.
    """
    return (x1.unsqueeze(-2) - x2.unsqueeze(-3)).square()


def rbf_kernel(r2: Tensor) -> Tensor:
    return torch.exp(r2.div(-2))


def rq_kernel(r2: Tensor, alpha: float | Tensor) -> Tensor:
    alpha = torch.as_tensor(alpha)
    return (1 + r2.div(2 * alpha)).pow(-alpha)

## The Matérn kernel at zero distance

This piece is worth reading even if you do not care about sparsity, because it applies
to any hand-rolled Matérn kernel written as a function of the squared distance.

The Matérn kernel is naturally expressed in terms of $r = \sqrt{r^2}$. But
$\frac{d}{dr^2}\sqrt{r^2} = \frac{1}{2\sqrt{r^2}}$ is singular at zero, so
differentiating through `r2.sqrt()` produces `NaN` on the diagonal of the kernel
matrix — where $r^2 = 0$ exactly. The gradient of the marginal likelihood then becomes
`NaN`, and the fit fails immediately.

The value of the kernel at zero is of course perfectly well-defined, and so are its
derivatives with respect to $r^2$; it is only this particular *code path* that is
singular. So we evaluate a Taylor expansion in $r^2$ at exactly $r^2 = 0$:

$$k(r^2) \approx 1 + \frac{\nu}{2(1-\nu)} r^2
  + \frac{\nu^2}{8(2 - 3\nu + \nu^2)} r^4 + O(r^6),$$

and use `torch.where` to select it only on the $r^2 = 0$ entries. Because the
expansion is evaluated *at* the expansion point, the returned value is exact — no
approximation error is introduced — but the new branch is twice differentiable. The
`masked_fill` on the other branch is what actually prevents the `NaN` from propagating
through the unused side of the `where`.

In [ ]:
def matern_kernel_at_zero(r2: Tensor, nu: Tensor) -> Tensor:
    """Taylor expansion of the Matern kernel around r2 = 0.

    Evaluated only *at* r2 = 0, where it is exact, but unlike the canonical
    expression in terms of r = sqrt(r2), it is twice differentiable there.
    """
    nu_r2 = nu * r2
    inv_taylor_coeffs = [2 * (1 - nu), 8 * (2 - 3 * nu + nu.pow(2))]
    result = torch.ones_like(nu_r2)
    for i, c in enumerate(inv_taylor_coeffs):
        result = result + nu_r2.pow(i + 1) / c
    return result


def matern_kernel(r2: Tensor, nu: float | Tensor) -> Tensor:
    """Matern kernel as a function of the squared distance, differentiable at zero.

    Args:
        r2: `n x m` tensor of squared distances.
        nu: Smoothness parameter, one of 0.5, 1.5, 2.5.

    Returns:
        The `n x m` kernel matrix.
    """
    nu = torch.as_tensor(nu).to(r2)
    # Mask the r2 == 0 entries out of the canonical branch so that the singular
    # derivative of sqrt at zero cannot propagate NaNs through torch.where.
    r = r2.masked_fill(r2 == 0, 1.0).sqrt()

    if nu == 0.5:
        polynomial = torch.ones_like(r)
    elif nu == 1.5:
        polynomial = (math.sqrt(3) * r).add(1)
    elif nu == 2.5:
        polynomial = (math.sqrt(5) * r).add(1).add(5.0 / 3.0 * r.square())
    else:
        raise NotImplementedError("nu must be one of 0.5, 1.5, 2.5.")

    covar = polynomial * torch.exp(-(2 * nu).sqrt() * r)
    return torch.where(r2 == 0, matern_kernel_at_zero(r2, nu), covar)

## The kernel

A note on the implementation. Ideally the per-dimension scaling would live in
`Kernel.__call__`, which already handles `active_dims`, so that every kernel could
inherit sparse ARD for free. That does not work cleanly today: `ScaleKernel` has no
lengthscale of its own, and composite kernels call the base kernel's `forward`
directly rather than its `__call__`, so the scaling would be silently skipped. This is
the same reason `active_dims` is currently only honored at the top level, and why
`ScaleKernel` copies `active_dims` from its base kernel. So instead we implement the
scaling in `forward` and take the base kernel's own lengthscale out of the picture
entirely by setting `has_lengthscale = False`.

In [ ]:
class SparseFeatureKernel(Kernel, RelevancePursuitMixin, ABC):
    """A stationary kernel whose per-dimension inverse squared lengthscales `beta`
    are an exactly sparse parameter, so that the set of active input dimensions can
    be discovered with Relevance Pursuit.

    Concrete subclasses supply `kernel_function`, a stationary kernel expressed as a
    function of the (weighted) squared distance.
    """

    has_lengthscale = False  # beta subsumes the base kernel's lengthscale

    def __init__(
        self,
        ard_num_dims: int,
        batch_shape: torch.Size | None = None,
        active_dims: tuple[int, ...] | None = None,
        beta_constraint: NonTransformedInterval | None = None,
        dtype: torch.dtype | None = None,
        device: torch.device | None = None,
        support: list[int] | None = None,
    ):
        """
        Args:
            ard_num_dims: The number of candidate features, i.e. the maximum
                support size.
            batch_shape: The batch shape of the kernel.
            active_dims: Dimensions to which the kernel is applied. The discovered
                support is a subset of these, so this acts as a hard prior.
            beta_constraint: Constraint on beta. Must be a `NonTransformedInterval`,
                since smooth transforms cannot represent exact zeros. A sensible
                upper bound is `1 / min_lengthscale**2`.
            support: Indices of the initially active features. Defaults to empty.
        """
        super().__init__(batch_shape=batch_shape, active_dims=active_dims)
        RelevancePursuitMixin.__init__(self, dim=ard_num_dims, support=support)

        if beta_constraint is None:
            beta_constraint = NonTransformedInterval(
                lower_bound=0.0, upper_bound=torch.inf, initial_value=0.0
            )
        if beta_constraint.lower_bound < 0:
            raise ValueError("SparseFeatureKernel requires a non-negative beta.")

        # The parameter is initialized in the sparse representation, which is what
        # RelevancePursuitMixin assumes.
        self.register_parameter(
            name="raw_beta",
            parameter=Parameter(
                torch.zeros(
                    *self.batch_shape, len(self.support), dtype=dtype, device=device
                )
            ),
        )
        self.register_constraint("raw_beta", beta_constraint)

    @property
    def sparse_parameter(self) -> Parameter:
        return self.raw_beta

    def set_sparse_parameter(self, value: Parameter) -> None:
        self.raw_beta = value.to(self.raw_beta)

    @property
    def beta(self) -> Tensor:
        return self.raw_beta_constraint.transform(self.raw_beta)

    @abstractmethod
    def kernel_function(self, r2: Tensor) -> Tensor:
        """The stationary kernel as a function of the weighted squared distance."""

    def forward(
        self,
        x1: Tensor,
        x2: Tensor,
        diag: bool = False,
        last_dim_is_batch: bool = False,
        **params,
    ) -> Tensor:
        r2_all = all_distances(x1, x2)  # n x m x d
        if self.is_sparse:
            # NOTE: index by self.support rather than by the boolean self.is_active,
            # so that the ordering matches that of the sparse parameter. is_active is
            # sorted by feature index, whereas the support is in insertion order.
            r2_all = r2_all[..., self.support]
        r2 = torch.einsum(r2_all, [..., 0, 1, 2], self.beta, [..., 2], [..., 0, 1])
        covar = self.kernel_function(r2)
        return covar.diagonal(dim1=-1, dim2=-2) if diag else covar


class SparseFeatureRBFKernel(SparseFeatureKernel):
    def kernel_function(self, r2: Tensor) -> Tensor:
        return rbf_kernel(r2)


class SparseFeatureMaternKernel(SparseFeatureKernel):
    def __init__(self, ard_num_dims: int, nu: float = 2.5, **kwargs):
        super().__init__(ard_num_dims=ard_num_dims, **kwargs)
        self.nu = nu

    def kernel_function(self, r2: Tensor) -> Tensor:
        return matern_kernel(r2, nu=self.nu)


class SparseFeatureRQKernel(SparseFeatureKernel):
    def __init__(self, ard_num_dims: int, alpha: float = 1.0, **kwargs):
        super().__init__(ard_num_dims=ard_num_dims, **kwargs)
        self.alpha = alpha

    def kernel_function(self, r2: Tensor) -> Tensor:
        return rq_kernel(r2, alpha=self.alpha)

## Test problems with known support

To measure support recovery we need problems whose ground-truth active set is known.
`EmbeddedTestFunction` embeds a low-dimensional test function into a
higher-dimensional space via a linear map; with an axis-aligned embedding the active
dimensions are exactly the first `base_function.dim` coordinates.

`manhattan_walk` generates designs that perturb only a sparse subset of coordinates
per step. The motivation is that when consecutive points differ in only a few
coordinates, the per-dimension distance matrices are less correlated with one another,
which should make the individual features easier to identify. Section E looks at that
claim more carefully.

In [ ]:
from botorch.test_functions.synthetic import SyntheticTestFunction


class EmbeddedTestFunction(SyntheticTestFunction):
    """Embeds a lower-dimensional test function into a higher-dimensional domain,
    so that the ground-truth set of relevant dimensions is known.
    """

    def __init__(
        self,
        base_function: SyntheticTestFunction,
        embedding: Tensor,
        noise_std: float | None = None,
        negate: bool = False,
        bounds: list[tuple[float, float]] | None = None,
    ) -> None:
        self.dim = embedding.shape[0]
        self._bounds = bounds or [(0.0, 1.0) for _ in range(self.dim)]
        super().__init__(noise_std=noise_std, negate=negate, bounds=bounds)
        self.base_function = base_function
        self.embedding = embedding  # dim x base_function.dim

    @property
    def _optimal_value(self) -> float:
        return self.base_function._optimal_value

    def evaluate_true(self, X: Tensor) -> Tensor:
        return self.base_function.evaluate_true(X @ self.embedding)


def manhattan_walk(
    n: int,
    d: int,
    num_pert: int = 1,
    bounds: Tensor | None = None,
    sigma: float = 0.05,
    uniform: bool = False,
) -> Tensor:
    """A random walk whose steps perturb only a sparse subset of the dimensions.

    Args:
        n: Number of points.
        d: Dimension.
        num_pert: Number of dimensions perturbed per step.
        bounds: `2 x d` bounds. Defaults to the unit cube.
        sigma: Step standard deviation, relative to the unit cube.
        uniform: If True, resample the perturbed coordinates uniformly instead of
            taking a Gaussian step. NOTE: this tends to be counterproductive, since
            large steps drive the kernel covariance to zero, at which point changes
            in the inputs no longer correlate with changes in the output.

    Returns:
        An `n x d` tensor of points.
    """
    X = torch.zeros(n, d)
    X[0] = torch.rand((d,))
    sigma = sigma / math.sqrt(num_pert)  # keep the total step norm constant
    for i in range(1, n):
        X[i, :] = X[i - 1, :]
        pert_indices = torch.randint(d, (num_pert,))
        if uniform:
            X[i, pert_indices] = torch.rand((num_pert,))
            continue
        delta = sigma * torch.randn((num_pert,))
        new_X = X[i, pert_indices] + delta
        out_of_bounds = (new_X < 0).logical_or(new_X > 1)
        while out_of_bounds.any():
            # Reflect and halve, which guarantees termination.
            delta = -delta / 2
            new_X[out_of_bounds] = (X[i, pert_indices] + delta)[out_of_bounds]
            out_of_bounds = (new_X < 0).logical_or(new_X > 1)
        X[i, pert_indices] = new_X.clamp_(min=0, max=1)

    if bounds is not None:
        X = (bounds[[1]] - bounds[[0]]) * X - bounds[[0]]
    return X

## Support recovery experiment

Hartmann-6 embedded in 32 dimensions, with 32 observations, under four design
distributions. The question is whether the recovered support contains the six
genuinely active dimensions.

In [ ]:
from botorch.models.transforms.outcome import Standardize
from botorch.test_functions.synthetic import Hartmann
from botorch.utils.sampling import draw_sobol_samples

torch.manual_seed(0)

base_function = Hartmann()
embedding_dim = 32
n = 32
active_dim = base_function.dim  # ground truth support is range(active_dim)

embedding = torch.vstack(
    (
        torch.eye(active_dim),
        torch.zeros(embedding_dim - active_dim, active_dim),
    )
)
objective = EmbeddedTestFunction(base_function=base_function, embedding=embedding)

bounds = torch.stack(
    (torch.zeros(embedding_dim), torch.ones(embedding_dim))
)

designs = {
    "Sobol": draw_sobol_samples(bounds=bounds, n=n, q=1).squeeze(-2),
    "Uniform": torch.rand(n, embedding_dim),
}
for num_pert in [1, 4, 8]:
    designs[f"Manhattan-{num_pert}"] = manhattan_walk(
        n, embedding_dim, num_pert=num_pert
    )

Ys = {}
for name, X in designs.items():
    Y = objective(X).unsqueeze(-1)
    Y = Y + 1e-2 * torch.randn_like(Y)
    Ys[name] = Standardize(m=1)(Y)[0]

In [ ]:
min_lengthscale = 0.1
max_beta = 1 / min_lengthscale**2


def make_models(X: Tensor, Y: Tensor) -> dict[str, SingleTaskGP]:
    d = X.shape[-1]
    likelihood = GaussianLikelihood(
        noise_constraint=NonTransformedInterval(1e-6, 1e-4, initial_value=5e-5)
    )
    kernels = {
        "ARD RBF": ScaleKernel(
            RBFKernel(
                ard_num_dims=d,
                lengthscale_constraint=NonTransformedInterval(
                    min_lengthscale, torch.inf, initial_value=1.0
                ),
            )
        ),
        "Sparse RBF": ScaleKernel(
            SparseFeatureRBFKernel(
                ard_num_dims=d,
                beta_constraint=NonTransformedInterval(
                    0.0, max_beta, initial_value=0.0
                ),
            ).full_support(),
            # The outputscale must be bounded away from zero: beta = 0 together with
            # outputscale = 0 is a stationary point that the optimizer can fall into.
            outputscale_constraint=NonTransformedInterval(
                0.01, 10.0, initial_value=1.0
            ),
        ),
    }
    return {
        name: SingleTaskGP(
            train_X=X,
            train_Y=Y,
            mean_module=ZeroMean(),
            covar_module=copy.deepcopy(kernel),
            likelihood=copy.deepcopy(likelihood),
        )
        for name, kernel in kernels.items()
    }

For the sparse model we use a *backward* schedule: start from the full support, fit,
contract to the target size, reset the parameter to zero, and refit on the reduced
support. Two details matter empirically.

First, backward tends to work better than forward here. Forward selection commits to
early choices, and with correlated features an early mistake is not recoverable;
starting from the full support avoids that. This mirrors the behavior reported for the
robust GP, and Section C shows a concrete example where forward fails and backward
succeeds.

Second, resetting `beta` to zero before the second fit matters more than one might
expect. After the first fit many entries sit exactly on a constraint boundary, and
restarting from there tends to leave them stuck.

In [ ]:
optimizer_kwargs = {"options": {"maxiter": 100_000, "ftol": 1e-10, "gtol": 1e-10}}
target_support_size = 16

results = {}
for name, X in designs.items():
    Y = Ys[name]
    models = make_models(X, Y)
    results[name] = {}
    for model_name, model in models.items():
        mll = ExactMarginalLogLikelihood(likelihood=model.likelihood, model=model)
        if model_name == "Sparse RBF":
            sparse_module = model.covar_module.base_kernel
            sparse_module.full_support()
            with torch.no_grad():
                sparse_module.sparse_parameter.zero_()
            fit_gpytorch_mll(mll, optimizer_kwargs=optimizer_kwargs)

            sparse_module.support_contraction(
                mll, n=embedding_dim - target_support_size
            )
            with torch.no_grad():
                sparse_module.sparse_parameter.zero_()
            sparse_module.to_sparse()
            fit_gpytorch_mll(mll, optimizer_kwargs=optimizer_kwargs)
        else:
            fit_gpytorch_mll(mll, optimizer_kwargs=optimizer_kwargs)

        results[name][model_name] = {
            "model": model,
            "mll": mll(model(X), Y.squeeze(-1)).item(),
        }

In [ ]:
dims = torch.arange(embedding_dim)
fig, axes = plt.subplots(
    len(designs), 2, figsize=(14, 3 * len(designs)), sharex=True
)

for row, name in enumerate(designs):
    sparse_model = results[name]["Sparse RBF"]["model"]
    sparse_kernel = sparse_model.covar_module.base_kernel
    beta = sparse_kernel.to_dense().beta.detach()

    ard_model = results[name]["ARD RBF"]["model"]
    inv_sq_lengthscale = (
        ard_model.covar_module.base_kernel.lengthscale.detach().pow(-2).squeeze(0)
    )

    for col, (values, label) in enumerate(
        [(beta, "Sparse RBF: beta"), (inv_sq_lengthscale, "ARD RBF: 1 / ell^2")]
    ):
        ax = axes[row, col]
        colors = ["tab:green" if i < active_dim else "tab:gray" for i in dims]
        ax.bar(dims, values, color=colors)
        ax.set_title(f"{name} — {label}")
        ax.set_ylabel("weight")

    sparse_kernel.to_sparse()

axes[-1, 0].set_xlabel("input dimension")
axes[-1, 1].set_xlabel("input dimension")
fig.suptitle("Green = genuinely active dimensions", y=1.0)
fig.tight_layout()

In [ ]:
# How much of the ground-truth support survived, and at what likelihood?
print(f"{'design':>14} | {'|S|':>4} | {'recovered':>9} | {'sparse MLL':>11} | {'ARD MLL':>9}")
for name in designs:
    sparse_kernel = results[name]["Sparse RBF"]["model"].covar_module.base_kernel
    support = set(sparse_kernel.support)
    recovered = len(support & set(range(active_dim)))
    print(
        f"{name:>14} | {len(support):>4} | {recovered:>4}/{active_dim} | "
        f"{results[name]['Sparse RBF']['mll']:>11.3f} | "
        f"{results[name]['ARD RBF']['mll']:>9.3f}"
    )

### Status & open problems

- **Marginal likelihood optimization is the blocker.** The support machinery behaves
  as intended, but the inner L-BFGS-B fits are unreliable: results are sensitive to
  the constraint bounds, to whether the parameter is reset between steps, and to the
  outputscale initialization. Until that is addressed the recovery numbers above
  should be read as a lower bound on what the parameterization can do. This is
  probably the single most valuable thing to fix.
- **Hessian preconditioning is untried.** The dimensions of `r2_all` have very
  different scales, so the Hessian in `beta` is poorly conditioned. Normalizing each
  `beta_i` by the corresponding per-dimension norm of `r2_all` should help, and the
  scaffolding is easy to add in `forward`, but it has not been evaluated.
- **The support-expansion criterion could be sharpened.** `support_expansion`
  currently uses the gradient at zero. A second-order correction — the gradient
  divided by a curvature estimate — would be closer to the closed-form relevance
  criteria used in Section B, and would account for the fact that features differ
  wildly in scale.
- **Collinear features are not handled.** If two dimensions have nearly collinear
  distance matrices, no amount of data distinguishes them, and greedy selection will
  pick one essentially at random. One could detect this up front and either drop one
  of the pair or average over both choices. Section E has the diagnostic; nothing
  acts on it yet.

<a id="b"></a>
# B. Sparse inducing points: the `RelevanceInducingKernel`

## Motivation

Sparse GP approximations replace the $n \times n$ kernel matrix with a low-rank
surrogate built from $m \ll n$ inducing points $Z$:

$$Q(X, X) = K(X, Z)\,\bigl(K(Z, Z) + \operatorname{diag}(\gamma)\bigr)^{-1} K(Z, X).$$

Choosing $m$ and the locations $Z$ is usually done by hand or by gradient descent on
continuous inducing locations. Relevance Pursuit offers a third option: fix a pool of
candidate inducing points, treat their inclusion as a sparse parameter, and let the
marginal likelihood decide how many to use and which.

## The inverse parameterization

There is a catch, and it is the same one as in Section A. In the expression above
$\gamma_i$ is a *variance* on the $i$-th inducing observation: $\gamma_i = 0$ means the
inducing observation is noiseless, and $\gamma_i \to \infty$ means it carries no
information and the point is effectively removed. Removal again sits at infinity.

Applying the Woodbury identity flips this around:

$$\bigl(K + D^{-1}\bigr)^{-1} = D - D\,\bigl(K^{-1} + D\bigr)^{-1} D,
  \qquad D = \operatorname{diag}(\gamma).$$

In this *inverse* parameterization $\gamma_i$ is a precision, and $\gamma_i = 0$ removes
the point exactly. That is the representation Relevance Pursuit needs.

The honest caveat: exact zeros make the inner matrix genuinely low-rank, and the
Cholesky factorizations below are not yet robust to that. Both parameterizations are
implemented so the two can be compared, but the inverse one is the interesting one and
the less stable one.

In [ ]:
from botorch.models.model import Model
from botorch.posteriors.gpytorch import GPyTorchPosterior
from gpytorch.distributions import MultivariateNormal
from linear_operator import to_dense
from linear_operator.operators import (
    DiagLinearOperator,
    LinearOperator,
    LowRankRootAddedDiagLinearOperator,
    LowRankRootLinearOperator,
    MatmulLinearOperator,
)
from linear_operator.utils.cholesky import psd_safe_cholesky


def inverse_root_via_cholesky(X: Tensor) -> Tensor:
    """Inverse of the upper Cholesky factor of `X`."""
    chol = psd_safe_cholesky(X, upper=True)
    eye = torch.eye(chol.size(-1), device=chol.device, dtype=chol.dtype)
    return torch.linalg.solve_triangular(chol, eye, upper=True)

In [ ]:
class RelevanceInducingKernel(Kernel, RelevancePursuitMixin):
    """An inducing point kernel whose inducing point precisions are an exactly sparse
    parameter, so that the number and location of inducing points can be discovered
    with Relevance Pursuit.
    """

    has_lengthscale = False

    def __init__(
        self,
        base_kernel: Kernel,
        inducing_points: Tensor,
        batch_shape: torch.Size | None = None,
        inducing_gamma: Tensor | None = None,
        inducing_gamma_constraint: NonTransformedInterval | None = None,
        inverse_parameterization: bool = True,
        diagonal_correction: bool = False,
        support: list[int] | None = None,
    ):
        """
        Args:
            base_kernel: The kernel to approximate.
            inducing_points: An `m x d` pool of candidate inducing points, in the
                model's transformed input space.
            inducing_gamma: Initial value of the sparse parameter.
            inducing_gamma_constraint: Constraint on gamma. Must be a
                `NonTransformedInterval` to permit exact zeros.
            inverse_parameterization: If True, gamma is a precision, so that
                gamma_i = 0 removes inducing point i. If False, gamma is a variance
                and removal corresponds to gamma_i -> infinity, which cannot be
                represented exactly. See the discussion above.
            diagonal_correction: If True, apply the fully independent conditional
                (FIC) diagonal correction. NOTE: this makes the *residual* covariance
                indefinite, so it must not be combined with `dtc_posterior`.
            support: Indices of the initially active inducing points.
        """
        super().__init__(batch_shape=batch_shape)
        dim = inducing_points.shape[-2]
        RelevancePursuitMixin.__init__(self, dim=dim, support=support)

        self.base_kernel = base_kernel
        if inducing_points.ndimension() == 1:
            inducing_points = inducing_points.unsqueeze(-1)
        # The candidate locations are held fixed; only their inclusion is learned.
        self.register_buffer("_inducing_points", inducing_points)

        dtype, device = inducing_points.dtype, inducing_points.device
        if inducing_gamma is None:
            inducing_gamma = torch.zeros(
                *self.batch_shape, len(self.support), dtype=dtype, device=device
            )
        self.register_parameter(
            name="raw_inducing_gamma", parameter=Parameter(inducing_gamma)
        )
        if inducing_gamma_constraint is None:
            inducing_gamma_constraint = NonTransformedInterval(
                lower_bound=0.0, upper_bound=torch.inf, initial_value=0.0
            )
        self.register_constraint("raw_inducing_gamma", inducing_gamma_constraint)

        self.inverse_parameterization = inverse_parameterization
        self.diagonal_correction = diagonal_correction

    @property
    def sparse_parameter(self) -> Parameter:
        return self.raw_inducing_gamma

    def set_sparse_parameter(self, value: Parameter) -> None:
        self.raw_inducing_gamma = value.to(self.raw_inducing_gamma)

    @property
    def inducing_gamma(self) -> Tensor:
        return self.raw_inducing_gamma_constraint.transform(self.raw_inducing_gamma)

    @property
    def inducing_points(self) -> Tensor:
        """The currently active inducing points."""
        return self._inducing_points[self.support]

    def _clear_cache(self) -> None:
        if hasattr(self, "_cached_kernel_inv_root"):
            del self._cached_kernel_inv_root

    @property
    def _inducing_matrix(self) -> Tensor:
        """Covariance between the active inducing inputs, or its inverse if
        `inverse_parameterization` is True.
        """
        Z = self.inducing_points
        K = to_dense(self.base_kernel(Z, Z))
        D = torch.diag_embed(self.inducing_gamma)
        if not self.inverse_parameterization:
            return K + D

        # Woodbury: (K + D^{-1})^{-1} = D - D (K^{-1} + D)^{-1} D
        K_inv_root = inverse_root_via_cholesky(K)
        K_inv = K_inv_root @ K_inv_root.transpose(-2, -1)
        K_inv_plus_D_root = inverse_root_via_cholesky(K_inv + D)
        K_inv_plus_D_inv = K_inv_plus_D_root @ K_inv_plus_D_root.transpose(-2, -1)
        return D - D @ K_inv_plus_D_inv @ D

    @property
    def _inducing_inv_root(self) -> Tensor:
        if not self.training and hasattr(self, "_cached_kernel_inv_root"):
            return self._cached_kernel_inv_root
        # In the inverse parameterization _inducing_matrix is already an inverse, so
        # we take its root rather than its inverse root. psd_safe_cholesky tolerates
        # the all-zeros case via its diagonal jitter.
        factorize = (
            psd_safe_cholesky
            if self.inverse_parameterization
            else inverse_root_via_cholesky
        )
        inv_root = factorize(self._inducing_matrix)
        if not self.training:
            self._cached_kernel_inv_root = inv_root
        return inv_root

    def forward(
        self, x1: Tensor, x2: Tensor, diag: bool = False, **kwargs
    ) -> Tensor | LinearOperator:
        k_x1z = to_dense(self.base_kernel(x1, self.inducing_points))

        if (x1 is x2) or torch.equal(x1, x2):
            covar = LowRankRootLinearOperator(k_x1z.matmul(self._inducing_inv_root))
            if self.diagonal_correction:
                # Fully independent conditional (FIC) correction; see eq. 25 of
                # Quinonero-Candela & Rasmussen (2005), JMLR 6:1939-1959.
                var = covar.diagonal(dim1=-1, dim2=-2)
                correction = self.base_kernel(x1, x2, diag=True) - var
                covar = LowRankRootAddedDiagLinearOperator(
                    covar, DiagLinearOperator(correction)
                )
        else:
            k_x2z = to_dense(self.base_kernel(x2, self.inducing_points))
            covar = MatmulLinearOperator(
                k_x1z.matmul(self._inducing_inv_root),
                k_x2z.matmul(self._inducing_inv_root).transpose(-1, -2),
            )

        return covar.diagonal(dim1=-1, dim2=-2) if diag else covar

    def num_outputs_per_input(self, x1: Tensor, x2: Tensor) -> int:
        return self.base_kernel.num_outputs_per_input(x1, x2)

    def residual_kernel(self, x1: Tensor, x2: Tensor, diag: bool = False) -> Tensor:
        """Residual between the base kernel and the inducing point approximation.

        This is the quantity that measures how much a candidate inducing point could
        still explain, and it is the basis of the relevance factors below.
        """
        kwargs = {"x1": x1, "x2": x2, "diag": diag}
        return to_dense(self.base_kernel(**kwargs) - self.forward(**kwargs))

### Closed-form relevance factors

The generic `support_expansion` criterion evaluates the gradient of the marginal
likelihood with respect to every inactive parameter, which here costs $O(n^2 m)$ per
candidate. For inducing points we can do better, because the optimal $\gamma$ for a
single candidate point is available in closed form — the same structure that
underlies sparse Bayesian learning and the relevance vector machine.

Writing $r_i = k(X, z_i) - q(X, z_i)$ for the residual against candidate $z_i$, and
$\Sigma$ for the current approximate covariance plus noise, define

$$q_i = r_i^\top \Sigma^{-1} y \quad \text{(quality)}, \qquad
  s_i = r_i^\top \Sigma^{-1} r_i \quad \text{(sparsity)}.$$

Adding $z_i$ improves the marginal likelihood exactly when $q_i^2 > s_i$, so
$q_i^2 / s_i - 1$ is a natural expansion objective and is what we override
`expansion_objective` with.

In [ ]:
class RelevanceInducingKernel(RelevanceInducingKernel):  # noqa: F811 (extend above)
    def expansion_objective(self, mll: ExactMarginalLogLikelihood) -> Tensor:
        """Score for each *inactive* candidate inducing point.

        Positive when adding the point is expected to improve the marginal
        likelihood, i.e. when the squared quality exceeds the sparsity factor.
        """
        quality, sparsity = self._compute_relevance_factors(mll)
        return quality.square() / sparsity - 1

    def _compute_relevance_factors(
        self, mll: ExactMarginalLogLikelihood
    ) -> tuple[Tensor, Tensor]:
        model = mll.model
        self._check_compatibility(model=model)
        # train() matters here: we want the prior from mll.model(X), whereas in eval()
        # __call__ returns the posterior.
        mll.train()
        X_untransformed = model.train_inputs[0]
        Y = model.train_targets
        X = model.input_transform(X_untransformed)
        F = model(X_untransformed)
        Sigma = mll.likelihood(F).covariance_matrix

        passive_points = self._inducing_points[~self.is_active]
        R = self.residual_kernel(X, passive_points)  # n x num_passive
        Sigma_inv_R = torch.linalg.solve(Sigma, R)

        sparsity = torch.einsum(R, [..., 0, 1], Sigma_inv_R, [..., 0, 1], [..., 1])
        quality = Sigma_inv_R.transpose(-2, -1) @ Y
        return quality, sparsity

    def _check_compatibility(self, model: Model) -> None:
        """The relevance factors above assume a specific model structure."""
        covar_module = model.covar_module
        is_covar = covar_module is self or (
            isinstance(covar_module, ScaleKernel) and covar_module.base_kernel is self
        )
        if not is_covar:
            raise ValueError(
                "This kernel must be the model's covar_module (optionally wrapped in "
                "a ScaleKernel) for the relevance factors to be valid."
            )
        if not isinstance(model.mean_module, ZeroMean):
            raise ValueError(
                "The relevance factors assume a zero mean function."
            )

    def dtc_posterior(
        self, model: Model, X: Tensor, alpha: float | Tensor | None = None
    ) -> GPyTorchPosterior:
        """Deterministic training conditional (DTC) correction to the posterior.

        Args:
            model: The model whose approximate posterior to compute.
            X: Test inputs.
            alpha: Strength of the correction, usually in [0, 1]. `alpha = 0` gives
                the uncorrected projected process approximation and `alpha = 1` the
                standard DTC correction; intermediate values interpolate.
        """
        if self.diagonal_correction:
            raise ValueError(
                "The DTC correction is not compatible with the FIC diagonal "
                "correction, since the resulting covariance need not be PSD. Set "
                "diagonal_correction=False, or use the uncorrected posterior."
            )
        self._check_compatibility(model=model)
        posterior = model.posterior(X)
        mean = posterior.distribution.loc  # unchanged by the correction
        sor_cov = posterior.distribution.covariance_matrix

        X_transformed = model.transform_inputs(X)
        res_cov = self.residual_kernel(X_transformed, X_transformed)
        if isinstance(model.covar_module, ScaleKernel):
            res_cov = res_cov * model.covar_module.outputscale

        if hasattr(model, "outcome_transform"):
            res_posterior = GPyTorchPosterior(
                MultivariateNormal(
                    mean=torch.zeros_like(mean), covariance_matrix=res_cov
                )
            )
            res_posterior = model.outcome_transform.untransform_posterior(
                res_posterior
            )
            res_cov = res_posterior.distribution.covariance_matrix

        cov = sor_cov + (res_cov if alpha is None else alpha * res_cov)
        return GPyTorchPosterior(
            MultivariateNormal(mean=mean, covariance_matrix=cov)
        )

### Demonstration

A one-dimensional sine, 32 observations, comparing an exact GP to the
relevance-inducing model. The thing to look at is where the inducing points end up and
how many survive.

In [ ]:
from botorch.models.transforms.input import Normalize
from gpytorch.kernels import MaternKernel

torch.manual_seed(0)

n, d = 32, 1
sigma = 1e-1
X_rik = torch.rand(n, d)
f = lambda X: torch.sin(2 * torch.pi * 2 * X)  # noqa: E731
Y_rik = f(X_rik) + sigma * torch.randn(n, d)

rik = RelevanceInducingKernel(
    base_kernel=MaternKernel(ard_num_dims=d),
    inducing_points=X_rik,  # candidate pool: the training inputs themselves
    inverse_parameterization=True,
).full_support()

# NOTE: do NOT wrap `rik` in a ScaleKernel here; the gammas already control the
# scale of the approximation, and the two would be redundant.
rik_model = SingleTaskGP(
    train_X=X_rik,
    train_Y=Y_rik,
    train_Yvar=sigma**2 * torch.ones_like(Y_rik),
    mean_module=ZeroMean(),
    covar_module=rik,
    input_transform=Normalize(d=d),
)
exact_model = SingleTaskGP(
    train_X=X_rik,
    train_Y=Y_rik,
    train_Yvar=sigma**2 * torch.ones_like(Y_rik),
    input_transform=Normalize(d=d),
)

for model in (exact_model, rik_model):
    fit_gpytorch_mll(
        ExactMarginalLogLikelihood(likelihood=model.likelihood, model=model)
    )

In [ ]:
X_plot = torch.linspace(0, 1, 512).unsqueeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(X_plot, f(X_plot), color="black", label="truth")
ax.plot(X_rik, Y_rik, "x", color="black", ms=8, label="data")
for model, label, color in [
    (exact_model, "exact GP", "tab:blue"),
    (rik_model, "relevance inducing", "tab:red"),
]:
    post = model.posterior(X_plot)
    mean = post.mean.detach().squeeze(-1)
    std = post.variance.sqrt().detach().squeeze(-1)
    ax.plot(X_plot, mean, label=label, color=color)
    ax.fill_between(
        X_plot.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.2, color=color
    )
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Posterior predictive")
ax.legend()

ax = axes[1]
gamma = rik.inducing_gamma.detach()
ax.plot(gamma.sort().values, torch.linspace(0, 1, len(gamma)))
ax.set_xlabel("inducing gamma")
ax.set_ylabel("cumulative fraction")
ax.set_title(f"Gamma distribution ({int((gamma > 0).sum())} of {n} active)")
fig.tight_layout()

### Empirical observations

Recorded from earlier runs of this setup; each is worth confirming.

- **The method adapts the number of inducing points to the noise level.** Because the
  gammas are learned through the marginal likelihood, a high noise variance lets the
  model summarize the data with fewer inducing points without sacrificing likelihood,
  while a low noise variance causes it to keep almost all of them. That seems like the
  right behavior, and it is not something you get from a fixed budget of $m$.
- **A Matérn base kernel prunes far more aggressively than an RBF.** A plausible
  explanation: the RBF spectrum decays so fast that dropping an inducing point barely
  moves the log-determinant, whereas the Matérn spectrum has a heavier tail, so each
  removal is more costly and the optimizer is pushed to be selective. This is a
  conjecture, not a result.
- **The posterior variance does not collapse at the inducing inputs**, in contrast to
  a canonical inducing point GP. The corollary is that the DTC-corrected posterior is
  usable directly, whereas the relevance vector machine typically needs transductive
  fixes such as adding test points to the inducing set.
- **On FITC and sample paths.** FITC can be made to correspond to a bona fide GP by
  adding a delta "kernel", but the resulting sample paths are non-smooth, which changes
  the modeling assumptions. It may be preferable to keep an approximate but genuine GP
  whose sample paths have the same differentiability as the kernel being approximated.

### Status & open problems

- **Low-rank incremental updates are unfinished.** Adding one inducing point is a
  rank-one update to the approximation, so evaluating a candidate should cost
  $O(nm + m^2)$ rather than the $O(nm^2 + m^3)$ of recomputing from scratch. The
  algebra is straightforward — the increment is $r_1 r_2^\top / r_{zz}$ in the notation
  above — but it is not wired into `expansion_objective`.
- **The Woodbury inverse hits a representational snag.** A diagonal approximation to
  the optimal gammas needs $\operatorname{diag}(\Sigma^{-1})$ for
  $\Sigma = A + UU^\top$. Woodbury gives
  $\Sigma^{-1} = A^{-1} - (A^{-1}UR)(A^{-1}UR)^\top$ with $RR^\top = (I + U^\top A^{-1} U)^{-1}$,
  which is a low-rank *subtraction* and therefore does not fit
  `LowRankRootAddedDiagLinearOperator`. Generalizing that operator to a signed
  Woodbury form would unlock this.
- **Numerical stability in the inverse parameterization.** Exact zeros make the inner
  matrix low-rank; `psd_safe_cholesky`'s jitter papers over this but is not a real fix.
  The empty-support case is not handled at all.
- **Cost of the relevance factors.** These are $O(n^2 k)$ over the passive set. A
  randomized low-rank approximation of the residual should help, possibly combined with
  only scoring a random subset of candidates, but the accuracy/cost tradeoff is
  unexplored.
- **No condensation step.** After fitting, the model could in principle be rewritten as
  a small exact GP over virtual observations at the inducing points, making posterior
  evaluation $O(m)$ and $O(m^2)$. Not implemented.

<a id="c"></a>
# C. Convexity and the optimization landscape

`SparseOutlierNoise` takes a `convex_parameterization` argument that defaults to
`True`, with the docstring noting that it "generally improves optimization results and
is thus recommended". This section is the reasoning behind that default.

Consider the negative log marginal likelihood as a function of a single robustness
variance $\rho$:

$$L(\rho) = \log\det(K + \rho) + y^\top (K + \rho)^{-1} y.$$

Relevance Pursuit needs to optimize this repeatedly, from the sparse solution
$\rho = 0$, so its behavior *near the origin* and the conditioning of its Hessian are
what determine whether the inner loop converges.

In [ ]:
from torch.autograd.functional import hessian, jacobian

K_scalar = 0.5 * torch.ones(1, 1)
y_scalar = torch.tensor([1.0])


def L(rho: Tensor, K: Tensor = K_scalar, y: Tensor = y_scalar) -> Tensor:
    """Negative log marginal likelihood, up to constants."""
    return (K + rho).logdet() + y @ (K + rho).inverse() @ y


rhos = torch.arange(0, 2.5, step=1 / 200)
values = torch.tensor([L(rho) for rho in rhos])
grads = torch.tensor([jacobian(L, rho) for rho in rhos])
hessians = torch.tensor([hessian(L, rho) for rho in rhos])

optimal_rho = (y_scalar - K_scalar).clamp(0)
convex_region = rhos[hessians >= 0]
convexity_boundary = convex_region.max().item() if len(convex_region) else rhos.max()
print(f"convex for rho < {convexity_boundary:.3f}; optimum at {optimal_rho.item():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rhos, values, label="value")
ax.plot(rhos, grads, label="gradient")
ax.plot(rhos, hessians, label="Hessian")
ax.plot(optimal_rho, L(optimal_rho), "o", color="black", ms=10, label="optimum")

ax.fill_between(
    [0, convexity_boundary], -2, 12, alpha=0.2, color="green", label="convex"
)
ax.fill_between(
    [convexity_boundary, rhos.max()], -2, 12, alpha=0.2, color="red", label="concave"
)
ax.set_xlabel("rho")
ax.set_ylabel("negative log marginal likelihood")
ax.set_title("The canonical parameterization is only locally convex")
ax.set_ylim(-2, 12)
ax.legend(ncol=2)

So the problem is convex in a neighborhood of the origin and concave beyond it. Where
the boundary falls depends on the data: the larger the residual, the further out the
optimum, and past a certain point the optimum lies in the concave region.

The Hessian has a compact closed form,
$H = \bigl(2\alpha\alpha^\top - K^{-1}\bigr) \odot K^{-1}$ with $\alpha = K^{-1} y$,
which we can check against autograd.

In [ ]:
a, b, num = 0.0, 1.6, 256
rho_grid = torch.linspace(a, b, num)
y_values = [torch.tensor([v]) for v in [0, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0]]

autograd_hessians = [
    torch.tensor([hessian(lambda r: L(r, y=y), rho) for rho in rho_grid])
    for y in y_values
]


def closed_form_hessian(rho: Tensor, K: Tensor, y: Tensor) -> Tensor:
    """H = (2 * alpha alpha^T - K^{-1}) o K^{-1}, with alpha = K^{-1} y."""
    K_inv = (K + rho).inverse()
    alpha = K_inv @ y
    return ((2 * alpha.outer(alpha) - K_inv) * K_inv).squeeze()


closed_form = [
    torch.tensor([closed_form_hessian(rho, K_scalar, y) for rho in rho_grid])
    for y in y_values
]
max_discrepancy = max(
    (h - c).abs().max().item() for h, c in zip(autograd_hessians, closed_form)
)
print(f"max |autograd - closed form| = {max_discrepancy:.2e}")

In [ ]:
cmap = plt.colormaps["plasma"]

fig, ax = plt.subplots(figsize=(8, 6))
for i, (h, y) in enumerate(zip(autograd_hessians, y_values)):
    color = cmap(i / len(autograd_hessians))
    ax.plot(rho_grid, h, color=color, label=f"|y| = {y.item():g}")

    rho_star = (y - K_scalar).clamp(0).view([1])
    ax.plot(
        rho_star, hessian(lambda r: L(r, y=y), rho_star), "o",
        color=color, ms=10, mec="black",
    )
    nonconvex = (h < 0).nonzero()
    if len(nonconvex):
        j = nonconvex[0]
        ax.plot(rho_grid[j], h[j], "p", color=color, ms=10, mec="black")

ax.axhline(0.0, color="black", lw=1)
ax.set_ylim(-5, 10)
ax.set_xlabel("rho")
ax.set_ylabel("second derivative")
ax.set_title("Circles: optimum.  Pentagons: onset of non-convexity.")
ax.legend(ncol=2)

## The convex reparameterization

We would like a change of variables $\rho = g(s)$ that keeps the origin fixed — so
that the sparse solution remains representable and $\rho$ retains its interpretation as
a robustness variance — while making the objective convex over the whole range. The
map

$$\rho(s) = \frac{1}{1-s} - 1, \qquad s \in [0, 1),$$

does this. It satisfies $g(0) = 0$, is monotone onto $[0, \infty)$, and pushes the
concave tail out to $s \to 1$ where it is compressed.

Several alternatives were tried and rejected: $\log\bigl(1/(1-s)\bigr)$,
$\exp\bigl(1/(1-s)\bigr) - 1$, $x \log x - x$, and powers $s^k$. The pattern is that
anything growing slower than cubically in $s$ near the origin fails to be convex there,
which is precisely the region that matters.

In [ ]:
EPS = 1e-6
convex_param = lambda s: 1 / (1 - s - EPS) - 1  # noqa: E731
inv_convex_param = lambda rho: 1 - 1 / (rho + 1)  # noqa: E731

s_grid = torch.linspace(0.0, 0.9, 256)


def L_convex(s: Tensor, K: Tensor = K_scalar, y: Tensor = y_scalar) -> Tensor:
    return L(convex_param(s), K=K, y=y)


convex_hessians = [
    torch.tensor([hessian(lambda s: L_convex(s, y=y), s) for s in s_grid])
    for y in y_values
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, curves, grid, xlabel, title in [
    (axes[0], autograd_hessians, rho_grid, "rho", "Canonical"),
    (axes[1], convex_hessians, s_grid, "s", "Convex reparameterization"),
]:
    for i, (h, y) in enumerate(zip(curves, y_values)):
        ax.plot(grid, h, color=cmap(i / len(curves)), label=f"|y| = {y.item():g}")
    ax.axhline(0.0, color="black", lw=1)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
axes[0].set_ylabel("second derivative")
axes[0].set_ylim(-2, 10)
axes[1].legend(ncol=2, fontsize="small")
fig.tight_layout()

### Preconditioning

In more than one dimension, being convex is not enough — the Hessian also has to be
reasonably conditioned, or the optimizer will crawl. Since the natural scale of
$\rho_i$ is set by the corresponding diagonal entry of $K$, scaling the map by
$\operatorname{diag}(K)$ is the obvious preconditioner:

$$\rho_i(s_i) = \left(\frac{1}{1-s_i} - 1\right) K_{ii}.$$

The heatmaps below show the Hessian condition number over a two-dimensional
$\rho$-grid for a deliberately ill-conditioned $K$, with non-convex regions masked out.

In [ ]:
import numpy as np
from matplotlib.colors import LogNorm

d2 = 2
K2 = torch.tensor([[0.5, 3e-2], [3e-2, 5e-2]])
y2 = torch.ones(d2)
num_grid = 128
grid_1d = torch.linspace(0.0, 3.0, num_grid)


def condition_numbers(param, inv_param) -> tuple[Tensor, Tensor]:
    latent_1d = inv_param(grid_1d)
    latent = torch.cartesian_prod(latent_1d, latent_1d)

    def L2(s: Tensor) -> Tensor:
        rho = torch.diag_embed(param(s))
        return (K2 + rho).logdet() + y2 @ (K2 + rho).inverse() @ y2

    H = torch.stack([hessian(L2, s) for s in latent])
    eigvals = torch.linalg.eigh(H).eigenvalues  # ascending
    cond = eigvals[:, -1] / eigvals[:, 0]
    is_convex = eigvals[:, 0] >= 0
    return cond.view(num_grid, -1), is_convex.view(num_grid, -1)


settings = {
    "Canonical": (lambda s: s, lambda rho: rho),
    "Convex + preconditioned": (
        lambda s: (1 / (1 - s + EPS) - 1) * K2.diagonal(),
        lambda rho: 1 - 1 / (rho + 1),
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (name, (param, inv_param)) in zip(axes, settings.items()):
    cond, is_convex = condition_numbers(param, inv_param)
    z = cond.clone()
    z[~is_convex] = np.nan  # non-convex regions are masked

    heat_cmap = plt.colormaps["viridis"].copy()
    heat_cmap.set_bad("red", 0.5)
    mesh = ax.pcolormesh(
        grid_1d, grid_1d, z, cmap=heat_cmap,
        norm=LogNorm(vmin=1.0, vmax=2e1), rasterized=True,
    )
    fig.colorbar(mesh, ax=ax)
    ax.set_xlabel("rho 1")
    ax.set_ylabel("rho 2")
    ax.set_title(f"{name} (red = non-convex)")
fig.suptitle("Hessian condition number")
fig.tight_layout()

## What this buys in practice

Fitting the robust GP on a 1-d sine with 8 corrupted observations, at a range of
L-BFGS-B convergence tolerances. The convex parameterization reaches substantially
better optima, and the gap grows as the tolerance tightens.

Values from an earlier run, reproduced here since this notebook is not executed:

| `ftol` | Canonical | Convex |
|---|---|---|
| 1e-03 | −4.37 | −14.18 |
| 1e-04 | −4.37 | −93.00 |
| 1e-05 | −4.37 | −93.01 |
| 1e-06 | −4.37 | −135.05 |
| 1e-07 | −98.62 | −518.68 |
| 1e-08 | −97.52 | −1139.29 |

The canonical parameterization stalls at the same value across four orders of
magnitude of tolerance, which is the signature of a badly conditioned problem rather
than of a converged one.

In [ ]:
import time

from botorch.models.likelihoods.sparse_outlier_noise import (
    SparseOutlierGaussianLikelihood,
    SparseOutlierNoise,
)
from botorch.models.relevance_pursuit import (
    backward_relevance_pursuit,
    forward_relevance_pursuit,
    get_posterior_over_support,
)
from botorch.models.transforms.outcome import Standardize
from gpytorch.likelihoods.noise_models import HomoskedasticNoise

# seed 1 produces a *cluster* of outliers, which is the interesting case below
torch.manual_seed(1)

n_c = 32
num_outliers = 8
f_c = lambda X: torch.sin(2 * torch.pi * 2 * X).sum(dim=-1, keepdim=True)  # noqa: E731

X_c = torch.rand(n_c, 1)
Y_c = f_c(X_c) + 2e-2 * torch.randn(n_c, 1)
Y_c[-num_outliers:] = 2 * torch.rand(num_outliers, 1) - 1  # corruptions

min_noise, max_noise = 1e-4, 1e-2
# The lengthscale must be bounded below: otherwise an arbitrarily wiggly GP can fit
# the corruptions exactly, and "outlier" stops being well defined.
min_lengthscale_c = 0.1


def make_robust_model(convex_parameterization: bool) -> SingleTaskGP:
    base_noise = HomoskedasticNoise(
        noise_constraint=NonTransformedInterval(
            min_noise, max_noise, initial_value=1e-3
        )
    )
    likelihood = SparseOutlierGaussianLikelihood(
        base_noise=base_noise,
        dim=X_c.shape[0],
        convex_parameterization=convex_parameterization,
    )
    covar_module = ScaleKernel(
        RBFKernel(
            ard_num_dims=1,
            lengthscale_constraint=NonTransformedInterval(
                min_lengthscale_c, torch.inf, initial_value=0.2
            ),
        ),
        outputscale_constraint=NonTransformedInterval(0.01, 10.0, initial_value=0.1),
    )
    return SingleTaskGP(
        train_X=X_c,
        train_Y=Y_c,
        mean_module=ZeroMean(),
        covar_module=covar_module,
        input_transform=Normalize(d=1),
        outcome_transform=Standardize(m=1),
        likelihood=likelihood,
    )

In [ ]:
timings, final_mlls = {"canonical": {}, "convex": {}}, {"canonical": {}, "convex": {}}
initial_rho = 1000.0  # deliberately far from the optimum

for name, convex in [("canonical", False), ("convex", True)]:
    for ftol in [1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8]:
        model = make_robust_model(convex_parameterization=convex)
        noise_covar = model.likelihood.noise_covar
        noise_covar.full_support()
        noise_covar.set_sparse_parameter(
            Parameter(torch.full((len(noise_covar.support),), initial_rho))
        )
        mll = ExactMarginalLogLikelihood(model.likelihood, model)
        mll.train()

        start = time.monotonic()
        fit_gpytorch_mll(
            mll,
            optimizer_kwargs={
                "options": {"maxiter": 10_000, "ftol": ftol, "gtol": 1e-8}
            },
        )
        timings[name][ftol] = time.monotonic() - start
        final_mlls[name][ftol] = mll(mll.model(X_c), Y_c.squeeze(-1)).item()

print(f"{'ftol':>8} | {'canonical':>10} | {'convex':>10}")
for ftol in final_mlls["canonical"]:
    print(
        f"{ftol:>8.0e} | {final_mlls['canonical'][ftol]:>10.2f} | "
        f"{final_mlls['convex'][ftol]:>10.2f}"
    )

## Forward versus backward selection

The seed above was chosen because the corruptions form a *cluster*. Clusters are the
known failure mode for forward, orthogonal-matching-pursuit-style selection: once a
couple of points in the cluster are admitted, the GP bends toward the cluster, and the
remaining corrupted points no longer look anomalous. Backward selection, which starts
from the full support and removes the least useful element, does not have this problem.

In [ ]:
fwd_model = make_robust_model(convex_parameterization=True)
fwd_mll = ExactMarginalLogLikelihood(fwd_model.likelihood, fwd_model)
fwd_module, fwd_trace = forward_relevance_pursuit(
    sparse_module=fwd_model.likelihood.noise_covar,
    mll=fwd_mll,
    sparsity_levels=list(range(n_c)),
    record_model_trace=True,
    reset_parameters=False,
)

bwd_model = make_robust_model(convex_parameterization=True)
bwd_mll = ExactMarginalLogLikelihood(bwd_model.likelihood, bwd_model)
bwd_module, bwd_trace = backward_relevance_pursuit(
    sparse_module=bwd_model.likelihood.noise_covar,
    mll=bwd_mll,
    sparsity_levels=list(range(n_c)),
    record_model_trace=True,
    reset_parameters=False,
)

### Choosing the support size

The traces give a sequence of models rather than a single answer. `get_posterior_over_support`
scores them by Bayesian model comparison against an exponential prior on the support
size, which both selects a model and quantifies how confident that selection is.

In [ ]:
prior_mean_of_support = 2.0

fwd_sizes, fwd_probs = get_posterior_over_support(
    SparseOutlierNoise, fwd_trace, prior_mean_of_support=prior_mean_of_support
)
bwd_sizes, bwd_probs = get_posterior_over_support(
    SparseOutlierNoise, bwd_trace, prior_mean_of_support=prior_mean_of_support
)
fwd_probs, bwd_probs = fwd_probs.detach(), bwd_probs.detach()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, sizes, probs, name in [
    (axes[0], fwd_sizes, fwd_probs, "forward"),
    (axes[1], bwd_sizes, bwd_probs, "backward"),
]:
    ax.bar(sizes, probs, color="tab:blue")
    ax.axvline(
        num_outliers, color="black", ls="--", label=f"true |S| = {num_outliers}"
    )
    ax.set_xlabel("support size")
    ax.set_title(f"{name}: MAP |S| = {sizes[probs.argmax()]}")
    ax.legend()
axes[0].set_ylabel("posterior probability")
fig.tight_layout()

In [ ]:
X_plot_c = torch.linspace(0, 1, 512).unsqueeze(-1)
viridis = plt.colormaps["viridis"]

fig, ax = plt.subplots(figsize=(9, 6))
plot_indices = [1, 2, 3, 4, 8, 16, 24, 30]
for i, idx in enumerate(plot_indices):
    model = bwd_trace[-idx]
    post = model.posterior(X_plot_c)
    mean = post.mean.detach().squeeze(-1)
    std = post.variance.sqrt().detach().squeeze(-1)
    color = viridis(1 - i / (2 * len(plot_indices)))
    support_size = len(model.likelihood.noise_covar.support)
    ax.plot(X_plot_c, mean, color=color, label=f"|S| = {support_size}")
    ax.fill_between(
        X_plot_c.squeeze(-1), mean - 2 * std, mean + 2 * std, alpha=0.15, color=color
    )

ax.plot(X_c[-num_outliers:], Y_c[-num_outliers:], "o", color="red", ms=9,
        label="corruptions")
ax.plot(X_c, Y_c, "o", color="black", ms=5)
ax.plot(X_plot_c, f_c(X_plot_c), ls="--", color="black", lw=2, label="truth")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_ylim(-1.2, 1.3)
ax.set_title("Backward relevance pursuit model trace")
ax.legend(ncol=3, fontsize="small")

## Appendix: the leave-one-out connection

There is a cheap diagnostic hiding in all of this. For a Gaussian process the
leave-one-out predictive mean and variance are available in closed form from a single
factorization of $\Sigma = K + \Sigma_{\text{noise}}$:

$$\sigma^2_{\text{LOO}, i} = \frac{1}{[\Sigma^{-1}]_{ii}}, \qquad
  \mu_{\text{LOO}, i} = y_i - \frac{[\Sigma^{-1} y]_i}{[\Sigma^{-1}]_{ii}}.$$

The optimal robustness variance for point $i$, holding everything else fixed, is then
just the excess of the squared LOO residual over the LOO variance:

$$\rho_i^\star = \bigl(\mu_{\text{LOO}, i} - y_i\bigr)^2 - \sigma^2_{\text{LOO}, i}.$$

This is the `loo=True` path in `SparseOutlierNoise`, and it is the reason the method is
affordable: the quantity that tells you which point to admit next costs one
factorization for all points at once. It is also a perfectly good standalone outlier
score, whether or not you go on to fit a robust model.

In [ ]:
def loo_statistics(
    model: SingleTaskGP, X: Tensor, Y: Tensor
) -> tuple[Tensor, Tensor, Tensor]:
    """Leave-one-out mean, variance, and the implied optimal rho."""
    X_transformed = model.transform_inputs(X)
    Y_transformed = Y
    if hasattr(model, "outcome_transform"):
        Y_transformed, _ = model.outcome_transform(Y)

    Sigma = to_dense(model.covar_module(X_transformed))
    Sigma = Sigma + model.likelihood.noise.detach() * torch.eye(len(X))
    Sigma_inv = Sigma.inverse()
    inv_diagonal = Sigma_inv.diagonal(dim1=-1, dim2=-2)

    y = Y_transformed.squeeze(-1)
    loo_var = 1 / inv_diagonal
    loo_mean = y - (Sigma_inv @ y) / inv_diagonal
    optimal_rho = (loo_mean - y).square() - loo_var
    return loo_mean, loo_var, optimal_rho


vanilla_model = SingleTaskGP(
    train_X=X_c,
    train_Y=Y_c,
    covar_module=ScaleKernel(
        RBFKernel(
            ard_num_dims=1,
            lengthscale_constraint=NonTransformedInterval(
                min_lengthscale_c, torch.inf, initial_value=0.2
            ),
        )
    ),
    input_transform=Normalize(d=1),
    outcome_transform=Standardize(m=1),
    likelihood=GaussianLikelihood(
        noise_constraint=NonTransformedInterval(min_noise, max_noise)
    ),
)
fit_gpytorch_mll(
    ExactMarginalLogLikelihood(vanilla_model.likelihood, vanilla_model)
)

loo_mean, loo_var, optimal_rho = loo_statistics(vanilla_model, X_c, Y_c)

fig, ax = plt.subplots(figsize=(9, 5))
indices = torch.arange(n_c)
colors = ["red" if i >= n_c - num_outliers else "tab:blue" for i in indices]
ax.bar(indices, optimal_rho.detach(), color=colors)
ax.axhline(0.0, color="black", lw=1)
ax.set_xlabel("observation index")
ax.set_ylabel("optimal rho")
ax.set_title("LOO-implied robustness variance (red = corrupted)")

### Status & open problems

- The convexity analysis is one-dimensional in $\rho$ and treats the other
  hyperparameters as fixed. The joint problem over $\rho$ *and* the kernel
  hyperparameters is the one actually being solved, and its landscape is not
  characterized.
- The choice $\rho(s) = 1/(1-s) - 1$ is convenient rather than derived. Whether some
  other map is better conditioned, or whether one can be derived from the geometry of
  the problem, is open.
- The preconditioning by $\operatorname{diag}(K)$ is a heuristic, motivated by the
  scalar case. A more principled diagonal preconditioner may exist.
- The empirical table above is a single problem instance at a single seed. It should
  be a proper sweep before anyone quotes it.

<a id="d"></a>
# D. A fixed-point iteration for the noise variance

**This one is unresolved**, and included as an open question rather than a result. The
iteration below has not been proven to be a contraction, and empirically its derivative
exceeds 1 in magnitude for several iterations when started from a large $\sigma^2$.

## Derivation

Eigendecompose the kernel matrix, $K = U \Lambda U^\top$, and project the observations
into the eigenbasis, $z = U^\top y$. The marginal likelihood becomes separable:

$$L(\sigma^2) = \sum_i \left[ \log(\lambda_i + \sigma^2)
   + \frac{z_i^2}{\lambda_i + \sigma^2} \right].$$

Setting the derivative to zero and rearranging gives a weighted average rather than a
closed form:

$$\sigma^2 = \frac{\sum_i w_i (z_i^2 - \lambda_i)}{\sum_i w_i},
  \qquad w_i = \frac{1}{(\lambda_i + \sigma^2)^2}.$$

The weights depend on $\sigma^2$, so this is a self-consistent equation, and the
natural thing to try is to iterate it. The appeal is that each step costs $O(n)$ once
the eigendecomposition is available, versus a full gradient step on the marginal
likelihood — and if it worked it might extend to the $\rho$ parameters, which would
make the inner loop of Relevance Pursuit much cheaper.

In [ ]:
def fixed_point_iterator(s2: Tensor, eigenvalues: Tensor, z: Tensor) -> Tensor:
    """One step of the noise variance fixed-point map."""
    w = 1 / (eigenvalues + s2).square()
    return (w * (z.square() - eigenvalues)).sum(-1) / w.sum(-1)


# Synthetic sanity check on a random spectrum.
torch.manual_seed(0)
n_fp = 1024
eigenvalues = 10 * torch.rand(n_fp).sort(descending=True).values
z = 20 * torch.rand(n_fp).sort(descending=True).values + torch.rand(())

s2 = torch.tensor(1e-2)
for i in range(16):
    s2_new = fixed_point_iterator(s2, eigenvalues, z).clamp(min=0)
    if i % 4 == 0 or i == 15:
        print(f"iter {i:>2}: s2 = {s2_new.item():.6f}  (step {(s2_new - s2).abs():.2e})")
    s2 = s2_new
s2_found = s2

## Does it contract?

A fixed-point iteration converges locally when $|f'(\sigma^2)| < 1$ near the fixed
point. Autograd gives us that derivative directly. Plotting $f$, the identity, and
$|f'|$ on log axes shows both where the fixed point is and where the map is actually
contractive.

In [ ]:
s2_grid = torch.logspace(-16, 10, steps=1024)
s2_grid.requires_grad = True
f_values = fixed_point_iterator(s2_grid.unsqueeze(-1), eigenvalues, z)
f_values.sum().backward()

derivative = s2_grid.grad.detach()
s2_plot = s2_grid.detach()
f_plot = f_values.detach()

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(s2_plot, f_plot.abs(), label="|f|")
ax.plot(s2_plot, derivative.abs(), label="|f'|")
ax.plot(s2_plot, s2_plot, ls="--", color="gray", label="identity")
ax.axhline(1.0, color="black", lw=1, label="contraction threshold")
ax.plot(s2_found.abs(), s2_found.abs(), "o", ms=10, label="fixed point found")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("sigma^2")
ax.set_title("The map is contractive only over part of the range")
ax.legend()

In [ ]:
# As sigma^2 -> infinity the weights flatten and the update tends to the
# unweighted mean of (z^2 - lambda).
w_infinity = 1 / eigenvalues.square()
print(f"unweighted mean:     {(z.square() - eigenvalues).mean():.6f}")
print(f"weight-limit value:  {((w_infinity / w_infinity.sum()) @ (z.square() - eigenvalues)):.6f}")

In [ ]:
# Validate against the marginal likelihood: is the gradient at the fixed point zero?
def log_marginal_likelihood(S: Tensor, y: Tensor) -> Tensor:
    return y @ torch.linalg.solve(S, y) + torch.logdet(S)


s2_check = s2_found.detach().clone().requires_grad_(True)
S_check = torch.diag(eigenvalues + s2_check)  # already in the eigenbasis
value = log_marginal_likelihood(S_check, z)
value.backward()
print(f"sigma^2 = {s2_check.item():.6e}, dL/d(sigma^2) = {s2_check.grad.item():.3e}")

### Open questions

- **Under what spectral conditions is the map a contraction?** The plot suggests it
  depends on where $\sigma^2$ sits relative to the bulk of the spectrum, but there is
  no analysis. A sufficient condition in terms of $\lambda$ and $z$ would settle
  whether this is usable.
- **Would damping help?** Replacing $\sigma^2 \leftarrow f(\sigma^2)$ with
  $\sigma^2 \leftarrow (1-\eta)\sigma^2 + \eta f(\sigma^2)$, or iterating in the log
  domain, would shrink the effective derivative and might widen the basin enough to
  matter.
- **Is it actually faster than L-BFGS-B on the marginal likelihood?** Never measured.
  The eigendecomposition is $O(n^3)$ up front, so the win would have to come from
  amortizing it across many inner solves — which is exactly the situation inside
  Relevance Pursuit.
- **Does it extend to the sparse parameters?** The $\rho$ updates in
  `SparseOutlierNoise` already have a leave-one-out closed form (Section C appendix).
  Whether a similar eigenbasis argument gives a cheap joint update for $\rho$ *and*
  $\sigma^2$ is the question that motivated this in the first place.

<a id="e"></a>
# E. Appendix: why the design distribution controls support recovery

Section A used several design distributions and got noticeably different results. This
appendix sketches why, borrowing the standard diagnostic from sparse recovery.

The sparse feature kernel sees the data only through the per-dimension squared
distances. Collect them into a matrix $L \in \mathbb{R}^{d \times \binom{n}{2}}$ whose
$i$-th row is the vectorized upper triangle of the pairwise squared differences in
dimension $i$. Fitting $\beta$ is then something close to a non-negative regression
problem against the rows of $L$, and the usual sparse-recovery intuition applies: if
two rows are nearly collinear, no amount of data distinguishes the corresponding
features, and a greedy method will pick between them essentially arbitrarily.

The standard measure is the mutual coherence, $\mu = \max_{i \neq j} |\langle
\tilde{L}_i, \tilde{L}_j\rangle|$ over normalized rows. Lower is better.

In [ ]:
def distance_feature_matrix(X: Tensor) -> Tensor:
    """Rows are the vectorized pairwise squared differences per dimension."""
    n_pts = X.shape[0]
    D = (X.unsqueeze(0) - X.unsqueeze(1)).square()  # n x n x d
    i, j = torch.triu_indices(n_pts, n_pts, offset=1)
    return D.permute(2, 0, 1)[..., i, j]  # d x n(n-1)/2


def coherence(L: Tensor) -> float:
    L_normalized = L / L.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    gram = L_normalized @ L_normalized.transpose(-2, -1)
    gram.fill_diagonal_(0.0)
    return gram.abs().max().item()


fig, ax = plt.subplots(figsize=(9, 6))
print(f"{'design':>14} | {'coherence':>9} | {'effective rank':>14}")
for name, X in designs.items():
    L = distance_feature_matrix(X)
    singular_values = torch.linalg.svdvals(L)
    normalized = singular_values / singular_values.max()
    effective_rank = int((normalized > 1e-8).sum())
    print(f"{name:>14} | {coherence(L):>9.4f} | {effective_rank:>14}")
    ax.plot(normalized, marker="o", ms=3, label=name)

ax.set_yscale("log")
ax.set_xlabel("index")
ax.set_ylabel("normalized singular value")
ax.set_title("Spectrum of the per-dimension distance features")
ax.legend()

### Open questions

- **Does coherence actually predict recovery here?** The table above and the recovery
  numbers in Section A can be correlated across seeds and designs; that has not been
  done. If the relationship is clean it would give a way to *choose* a design before
  collecting data.
- **The Babel function is probably the better measure.** Coherence only looks at the
  worst pair. The Babel function, which bounds the worst-case correlation between one
  row and any $k$ others, is the natural generalization when the true support has size
  $k$, and would likely correlate better.
- **A conjecture about the Manhattan walk.** The number of coordinates perturbed per
  step should directly control coherence: perturbing one coordinate at a time makes the
  distance features nearly disjoint, while perturbing all of them makes the walk look
  like uniform sampling. If so, the degree of parallelism in the walk is the knob that
  sets recoverability, and there may be an optimal setting balancing coherence against
  how efficiently the design explores the space.
- **Acting on the diagnostic.** If two rows are detected as collinear before fitting,
  one could drop one of the pair, reducing to an easier subproblem, and optionally
  average predictions over the choice of which to keep. Nothing does this yet.

---

## References

- S. Ament, E. Santorella, D. Eriksson, B. Letham, M. Balandat, E. Bakshy.
  [Robust Gaussian Processes via Relevance Pursuit](https://arxiv.org/abs/2410.24222).
  Advances in Neural Information Processing Systems 37, 2024.
- M. Tipping, A. Faul. Fast Marginal Likelihood Maximisation for Sparse Bayesian
  Models. AISTATS, 2003.
- J. Quiñonero-Candela, C. E. Rasmussen. A Unifying View of Sparse Approximate
  Gaussian Process Regression. JMLR 6:1939–1959, 2005.